# Workflow & Runner — End-to-End with Safe Formulas

This notebook builds a 4-step pipeline on a tiny dataset and runs each step
through `safe_formula.run_formula`. Each step's output becomes a step variable
available to the next, exactly mirroring how the backend engine sequences a
project file.

Pipeline:

1. **step1** — source data (a small list of articles)
2. **step2** — extract the word count of each title
3. **step3** — tag each title with `«…»` brackets
4. **step4** — derive a `is_long_title` flag using a literal threshold


In [1]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
from SIMPLE_STEPS.decorators import simple_step
from SIMPLE_STEPS.safe_formula import run_formula, validate, FormulaError



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py",

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py",

AttributeError: _ARRAY_API not found

## Step library

Register the operations our workflow needs. Each is a plain Python function
with a type-hinted signature — that's all the metadata the runner needs to
dispatch a call from a formula.


In [2]:
@simple_step(id="wf_word_count", category="Workflow")
def wf_word_count(text: str) -> int:
    """Number of whitespace-separated tokens in `text`."""
    return len(str(text).split())

@simple_step(id="wf_tag", category="Workflow")
def wf_tag(text: str, prefix: str = "«", suffix: str = "»") -> str:
    return f"{prefix}{text}{suffix}"

@simple_step(id="wf_threshold", category="Workflow")
def wf_threshold(value: int, cutoff: int) -> bool:
    """True if value >= cutoff."""
    return int(value) >= int(cutoff)


## The workflow definition

A workflow here is just an ordered list of `(step_name, formula)` pairs. This
is a deliberately small surface — the backend engine in `engine.py` uses the
same shape, layered on top of session/storage management.


In [3]:
workflow = [
    ("step2", "=wf_word_count(text=step1.title)"),
    ("step3", "=wf_tag(text=step1.title, prefix='«', suffix='»')"),
    ("step4", "=wf_threshold(value=step2.wf_word_count_output, cutoff=3)"),
]
workflow


[('step2', '=wf_word_count(text=step1.title)'),
 ('step3', "=wf_tag(text=step1.title, prefix='«', suffix='»')"),
 ('step4', '=wf_threshold(value=step2.wf_word_count_output, cutoff=3)')]

## The runner

A runner is just a loop. For each step:

1. Validate the formula against the currently available step names.
2. If invalid, halt with a structured error (this is what the API would surface
   to the UI as a red squiggle, never executing anything).
3. Otherwise, `run_formula` the expression and store the resulting DataFrame
   under the step's name so later steps can reference it.


In [4]:
def run_workflow(source_df: pd.DataFrame, workflow: list[tuple[str, str]]) -> dict:
    """
    Execute an ordered list of (step_name, formula) pairs.
    Returns a dict {step_name: DataFrame}.
    """
    steps: dict[str, pd.DataFrame] = {"step1": source_df}

    for step_name, formula in workflow:
        available = set(steps.keys())
        errors = validate(formula, available_steps=available)
        if errors:
            raise FormulaError(
                f"Step '{step_name}' formula failed validation:\n  - "
                + "\n  - ".join(errors)
            )

        print(f"▶ {step_name}: {formula}")
        result = run_formula(formula, steps=steps)
        # Normalize: run_formula may return StepProxy, DataFrame, or scalar.
        if hasattr(result, "df"):
            out_df = result.df
        elif isinstance(result, pd.DataFrame):
            out_df = result
        else:
            out_df = pd.DataFrame([{f"{step_name}_value": result}])
        steps[step_name] = out_df
        print(f"   → columns: {list(out_df.columns)}, rows: {len(out_df)}")
    return steps


## Run it


In [5]:
source = pd.DataFrame({
    "title": [
        "intro to pandas",
        "advanced sql for analytics",
        "hi",
        "a complete guide to safe formula interpreters",
    ],
})

results = run_workflow(source, workflow)


▶ step2: =wf_word_count(text=step1.title)
   → columns: ['title', 'wf_word_count_output'], rows: 4
▶ step3: =wf_tag(text=step1.title, prefix='«', suffix='»')
   → columns: ['title', 'wf_tag_output'], rows: 4
▶ step4: =wf_threshold(value=step2.wf_word_count_output, cutoff=3)
   → columns: ['title', 'wf_word_count_output', 'wf_threshold_output'], rows: 4


In [6]:
results["step2"]


,title,wf_word_count_output
0,intro to pandas,3
1,advanced sql for analytics,4
2,hi,1
3,a complete guide to safe formula interpreters,7


In [7]:
results["step3"]


,title,wf_tag_output
0,intro to pandas,«intro to pandas»
1,advanced sql for analytics,«advanced sql for analytics»
2,hi,«hi»
3,a complete guide to safe formula interpreters,«a complete guide to safe formula interpreters»


In [8]:
results["step4"]


,title,wf_word_count_output,wf_threshold_output
0,intro to pandas,3,True
1,advanced sql for analytics,4,True
2,hi,1,False
3,a complete guide to safe formula interpreters,7,True


## What happens when a step is broken?

Suppose someone edits step 3 to reference an op that doesn't exist. The
runner halts at validation — `run_formula` is never invoked, so no Python
function actually runs.


In [9]:
broken_workflow = [
    ("step2", "=wf_word_count(text=step1.title)"),
    ("step3", "=delete_everything(path='/')"),       # not registered
    ("step4", "=wf_threshold(value=step2.wf_word_count_output, cutoff=3)"),
]

try:
    run_workflow(source, broken_workflow)
except FormulaError as e:
    print("Workflow halted as expected:\n")
    print(e)


▶ step2: =wf_word_count(text=step1.title)
   → columns: ['title', 'wf_word_count_output'], rows: 4
Workflow halted as expected:

Step 'step3' formula failed validation:
  - Unknown operation: 'delete_everything'
  - Unknown name: 'delete_everything'


## Where this fits

Everything above mirrors what `SIMPLE_STEPS/engine.py` does on the backend
for a saved project file — sequence steps, resolve references, dispatch to
the registry — minus the session/storage layer. The key property is that the
**formula string is the canonical artifact**: the UI edits it, the runner
parses and validates it, and the same string round-trips into the project
file on disk.
